# 05 · Complejidad, despliegue y cierre de entregas

## Por qué existe este cuaderno
El documento del reto reparte la nota así: **desempeño 30 %** · **despliegue 25 %** · **eficiencia y complejidad 15 %** ·
pipeline y análisis 20 % · comunicación 10 %. La **Etapa 4** pide reportar, para cada modelo: **número de parámetros,
tamaño en disco (MB) y tiempo de inferencia por imagen**, y la **Entrega 3** pide **comparar desempeño y complejidad entre versiones**.

## Cómo se mide para que sea honesto
* Se descartan las primeras llamadas (calentamiento) y se reporta **mediana y p95**, no solo el promedio.
* Se mide **una imagen por llamada**: es lo que vive el usuario de la app. En lote es más rápido por imagen, pero no es el caso real.
* **Una CPU de Colab no es un celular.** Estas cifras comparan versiones entre sí. El tiempo en un celular real se mide
  en el celular y se reporta aparte (paso 4).

In [ ]:
# ===================== CELDA DE ARRANQUE: no hay que editar nada =====================
# Monta tu Drive, encuentra sola la carpeta del proyecto y Surface Crack, y deja todo listo.
from google.colab import drive
drive.mount("/content/drive")

import glob, os, sys
_raiz = "/content/drive/MyDrive"
_cands = sorted(glob.glob(f"{_raiz}/*/src/datos.py") + glob.glob(f"{_raiz}/*/*/src/datos.py"),
                key=os.path.getmtime, reverse=True)
if not _cands:
    raise FileNotFoundError("No encontré la carpeta del proyecto NUEVO en tu Drive (busco una carpeta que tenga src/datos.py). "
                            "Subí la carpeta del zip nuevo a tu Drive y volvé a ejecutar esta celda.")
PROYECTO = os.path.dirname(os.path.dirname(_cands[0]))
os.chdir(PROYECTO)
sys.path.insert(0, PROYECTO)
print("Proyecto:", PROYECTO)
!pip -q install scikit-image scikit-learn pandas matplotlib

from src import colab
RUTA_DATOS = colab.preparar(PROYECTO, _raiz)       # Surface Crack (zip -> disco local, o carpeta en Drive)

import json, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from src import config, datos, evaluar, entrenar, modelo, complejidad, inclinacion, informe, verificar

from tensorflow import keras
te = pd.read_csv("results/particion_prueba_verificada.csv" if Path("results/particion_prueba_verificada.csv").exists() else "results/particion_prueba.csv")

## Paso 1 · Tres variantes TFLite por modelo
**Qué hace:** exporta cada modelo en tres formatos y mide tamaño, velocidad y **fidelidad** (si la conversión cambió las respuestas).

| Variante | Qué hace | Esperable |
|---|---|---|
| float32 | sin comprimir | referencia |
| float16 | pesos en 16 bits | ~2× más chico, casi sin pérdida |
| dinámico | pesos en 8 bits | ~4× más chico; **verificar** que no cambie respuestas |

**Qué decidir:** la variante más chica y rápida cuya fidelidad sea de **0 cambios de clase**.

In [ ]:
MODELOS = {n: a for n, a in {"v2_mobilenet": "mobilenetv2", "v2_efficientnet": "efficientnetb0"}.items() if Path(f"models/{n}/modelo.keras").exists()}
muestra = entrenar.cargar_imagenes(te.sample(min(300, len(te)), random_state=1)["ruta"].tolist())
filas, fidelidad = [], {}
for nombre, arq in MODELOS.items():
    m = keras.models.load_model(f"models/{nombre}/modelo.keras")
    inf = modelo.modelo_inferencia(m, arq)
    variantes = modelo.exportar_variantes(m, f"models/{nombre}", arq)
    for var, (ruta, _) in variantes.items():
        fila = complejidad.fila_complejidad(f"{nombre}/{var}", modelo_keras=m, modelo_inf=inf,
                                            ruta_keras=f"models/{nombre}/modelo.keras", ruta_tflite=ruta)
        fid = complejidad.fidelidad_tflite(entrenar.predictor(inf), complejidad.tflite_predictor(ruta), muestra)
        fila["cambian_de_clase_vs_keras"] = fid["cambian_de_clase"]
        fila["dif_max_prob"] = round(fid["dif_max"], 4)
        filas.append(fila)
        fidelidad[f"{nombre}/{var}"] = fid
tabla_comp = complejidad.tabla_complejidad(filas)
display(tabla_comp)
tabla_comp.to_csv("results/complejidad.csv", index=False)
Path("results/fidelidad_tflite.json").write_text(json.dumps(fidelidad, indent=2), encoding="utf-8")

## Paso 2 · Comparación entre versiones (Entrega 3)
Une, por versión: complejidad y desempeño en **tres conjuntos**: la prueba sin fuga, la fuente fuera de dominio y las fotos propias.

⚠ **La v1 (modelo anterior) NO se puede evaluar en la prueba nueva:** se entrenó con una división imagen por imagen, así que
vio muchas de esas imágenes. Compararla ahí sería injusto. Se compara **solo** en fuera de dominio y fotos propias, que ninguna
versión vio al entrenar. Si no tenés `models/mobilenet_100ep.tflite`, la fila de v1 se omite.

In [ ]:
fuera = pd.read_csv("results/fuera_de_dominio.csv") if Path("results/fuera_de_dominio.csv").exists() else pd.DataFrame()
fp = pd.read_csv("results/fotos_propias_analisis.csv") if Path("results/fotos_propias_analisis.csv").exists() else pd.DataFrame()
fp_et = fp.dropna(subset=["etiqueta"]) if len(fp) else fp

def en_conjunto(predictor, df, col_ruta="ruta", carpeta=None, n=1500):
    if len(df) == 0:
        return {}
    df = df.sample(min(n, len(df)), random_state=1)
    rutas = [f"{carpeta}/{a}" for a in df[col_ruta]] if carpeta else df[col_ruta].tolist()
    return evaluar.metricas(df["etiqueta"].to_numpy(), entrenar.predecir_rutas(predictor, rutas))

lb = pd.read_csv("results/linea_base.csv")
mejor_lb = lb.sort_values("recall", ascending=False).iloc[0]
versiones = [{"version": "línea base: " + mejor_lb["metodo"], "exactitud_prueba": mejor_lb["exactitud"], "recall_prueba": mejor_lb["recall"]}]

candidatos = []
if Path("models/mobilenet_100ep.tflite").exists():
    candidatos.append(("v1 MobileNetV2 (división con fuga)", "models/mobilenet_100ep.tflite", None))
for nombre, arq in MODELOS.items():
    candidatos.append((f"v2 {arq}", f"models/{nombre}/modelo_float16.tflite", nombre))

for etiqueta, ruta_tflite, nombre in candidatos:
    pred = complejidad.tflite_predictor(ruta_tflite)
    fila = {"version": etiqueta}
    fila.update({k: v for k, v in complejidad.fila_complejidad(etiqueta, ruta_tflite=ruta_tflite, n=50).items() if k != "modelo"})
    if nombre:   # solo las v2 se evalúan en la prueba sin fuga
        r = en_conjunto(pred, te)
        fila.update({"exactitud_prueba": round(r["exactitud"], 4), "recall_prueba": round(r["recall"], 4)})
        fila["parametros"] = complejidad.contar_parametros(keras.models.load_model(f"models/{nombre}/modelo.keras"))["parametros"]
    ro = en_conjunto(pred, fuera)
    if ro:
        fila.update({"exactitud_fuera_dominio": round(ro["exactitud"], 4), "recall_fuera_dominio": round(ro["recall"], 4)})
    rf = en_conjunto(pred, fp_et, col_ruta="archivo", carpeta="data/fotos_propias")
    if rf:
        fila.update({"exactitud_fotos_propias": round(rf["exactitud"], 4), "recall_fotos_propias": round(rf["recall"], 4)})
    versiones.append(fila)
comparacion = pd.DataFrame(versiones)
display(comparacion)
comparacion.to_csv("results/comparacion_versiones.csv", index=False)

## Paso 3 · Inclinación sobre fotos propias (Entrega 1)
**Qué hace:** estima el desaplome de cada foto con Canny + Hough. Si diste el giro del celular (`roll`) lo descuenta; sin él, la medición
queda como *ángulo en la imagen* y no como inclinación real del muro.

**Cómo verificarlo a ojo:** para un muro que sabés a plomo, el ángulo corregido debe ser ≈ 0°.

In [ ]:
import cv2
roll = {}   # {"foto03.jpg": 3.0}  giro del celular (°) por foto, del sensor de gravedad
filas = []
fotos = sorted(Path("data/fotos_propias").glob("*.jp*g")) + sorted(Path("data/fotos_propias").glob("*.png"))
for ruta in fotos:
    est = inclinacion.estimar_inclinacion(cv2.imread(str(ruta), cv2.IMREAD_GRAYSCALE))
    fila = {"archivo": ruta.name, "angulo_imagen_deg": None, "n_lineas": 0, "inclinacion_real_deg": None, "roll_deg": roll.get(ruta.name)}
    if est:
        fila.update({"angulo_imagen_deg": round(est["angulo_imagen_deg"], 2), "n_lineas": est["n_lineas"]})
        if ruta.name in roll:
            fila["inclinacion_real_deg"] = round(inclinacion.corregir_con_gravedad(est["angulo_imagen_deg"], roll[ruta.name])["inclinacion_real_deg"], 2)
    filas.append(fila)
incl = pd.DataFrame(filas)
display(incl)
incl.to_csv("results/inclinacion_fotos_propias.csv", index=False)

## Paso 4 · Medir en el celular (lo que la CPU de Colab no puede decir)
Para el criterio de eficiencia conviene un número **del celular**. Dos caminos:
1. En la app Android, medir con `System.nanoTime()` alrededor de `interpreter.run(...)`, descartando las primeras 10 llamadas y promediando 100.
2. Con la herramienta *TFLite benchmark model* sobre el `.tflite` en el dispositivo.

Reportá el **modelo del celular**, la **cantidad de hilos** y si usó GPU/NNAPI. Anotalo en `results/tiempo_celular.txt`.

## Paso 5 · Cerrar: informe y estado de las entregas
* `informe.main()` escribe `results/secciones_informe_v2.md` **desde los CSV reales**: tablas de resultados, complejidad, versiones y fotos
  propias, y las reglas EXACTAS del motor de riesgo. Se copia a `INFORME_TECNICO.md` (secciones 6, 8, 9 y 10).
* `verificar.main()` marca cada requisito del documento del reto **según la evidencia que exista**. Lo que no tiene evidencia queda en ❌
  y dice qué falta.

In [ ]:
informe.main(".")
estado = verificar.main(".")